# EXP-2026-001 on Colab: Stage 4 overfit check on 16 training slices

The settings and pass rules were fixed in `configs/experiments/EXP-2026-001.json` (block `stage4_overfit`) before this run. Every step calls tested code in `rhuh_synthesis/overfit.py`, `training.py` and `stage3.py`:

1. **Preflight**: the 4 -> 11 channel adapter must match the released model with TF32 off (Stage 3 left a 0.0024 difference, likely TF32 rounding).
2. **Select 16 training slices** from 4 accepted training patients: 2 with residual tumor (2 ID-3 slices + 2 label-free slices each) and 2 without (2 labelled + 2 label-free slices each).
3. **Evaluate at step 0** (untrained adapter), **train 500 optimizer steps** (bf16, 4 pairs x 5 accumulation = 20 pairs per step, lr 1e-5), evaluate again at steps 250 and 500.
4. Each evaluation runs a fixed-noise loss per stream and one DDIM generation per slice (50 steps, guidance 7.5, seed 11), scored against its own training target. At step 500 a **shifted-condition control** (pre-op inputs from another patient) runs too.
5. **Decision**: PASS or ITERATE, by the preregistered checks.

Training data only; validation and test stay untouched. A pass only shows the pipeline can fit 16 slices. It is not a quality, generalization or clinical result. Expect about 35 minutes on the RTX PRO 6000 (GPU runtime). Outputs are written once to `synthesis/EXP-2026-001/stage4/<RUN_TAG>/` on Drive. The PNG grids there show patient images: keep them private.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN_TAG = "r1"                 # use a new tag for every attempt; a run folder is never reused
SAVE_FINAL_CHECKPOINT = False  # True writes an 11 GB checkpoint to /content (not Drive)

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import copy
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
RUN = PRIVATE / "stage4" / RUN_TAG
assert not RUN.exists(), f"{RUN} exists: choose a new RUN_TAG (records are never overwritten)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
STARTED = time.perf_counter()

## 2. Models: MedSegFactory checkpoint (strict), frozen VAE and CLIP, schedulers

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL, DDPMScheduler
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import reference_ddim_scheduler
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
S4 = config["stage4_overfit"]
assert S4["evaluate_at"][0] == 0 and S4["evaluate_at"][-1] == S4["steps"]
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
ddim = reference_ddim_scheduler()
record = {
    "stage": "4_overfit", "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "config_sha256": config_sha256(config), "stage4_config": S4, "environment": environment(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "checkpoint": {"repo": MSF_REPO, "file": MSF_FILE, "sha256": file_sha256(checkpoint),
                   "missing_keys": len(load_report["missing_keys"]), "unexpected_keys": len(load_report["unexpected_keys"])},
}
print({k: record[k] for k in ("commit", "config_sha256", "torch", "cuda", "gpu")}, record["checkpoint"])

## 3. Preflight: adapter equivalence with TF32 off (and on, for reference)

In [ ]:
from rhuh_synthesis.stage3 import adapter_equivalence

limits = S4["preflight_adapter_equivalence"]
record["equivalence_tf32_off"] = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=False)
record["equivalence_tf32_on"] = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=True)
off = record["equivalence_tf32_off"]
record["preflight_pass"] = bool(off["relative_difference"] <= limits["relative_difference_max"]
                                and off["copied_weights_identical"] and off["added_weights_zero"])
print("TF32 off:", off)
print("TF32 on: ", record["equivalence_tf32_on"])
assert record["preflight_pass"], "Adapter preflight failed: stop here and send the two lines above"


## 4. Select the 16 training slices and cache their features

In [ ]:
from rhuh_synthesis.overfit import label_volumes, ranked_patients, restrict, select_overfit_set
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.stage3 import clip_encoder
from rhuh_synthesis.training import build_feature_cache
from rhuh_synthesis.working_grid import to_working_grid

split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
train_patients = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
paths = {p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in train_patients}
volumes = {}
for patient, path in paths.items():
    with np.load(path) as data:
        volumes[patient] = label_volumes(data["label"])
ranked = ranked_patients(volumes)


def load_grid(patient):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    crop, grid = to_working_grid(arrays)
    grid["post_support"] = crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)
    return grid


selection, grids, skipped = select_overfit_set(ranked, load_grid, positive=S4["patients"]["positive"],
                                               negative=S4["patients"]["negative"], **S4["slices_per_patient"])
patients = [entry["patient"] for entry in selection]
exports = {entry["patient"]: restrict(grids[entry["patient"]], [z for z, _ in entry["slices"]]) for entry in selection}
items = [(entry["patient"], k) for entry in selection for k in range(len(entry["slices"]))]
categories = [category for entry in selection for _, category in entry["slices"]]
letters = {entry["patient"]: "ABCD"[n] for n, entry in enumerate(selection)}
names = [f"{letters[p]}-{c}-{k + 1}" for (p, k), c in zip(items, categories)]
targets = [{"post_image": exports[p]["post_image"][k], "pre_image": exports[p]["pre_image"][k],
            "label": exports[p]["label"][k], "region": exports[p]["pre_foreground"][k] & exports[p]["post_support"][k]}
           for p, k in items]
pixel_mm = config["representation"]["crop_mm"] / config["representation"]["out_px"]
del grids
cache = build_feature_cache(exports, vae, clip_encoder(tokenizer, text_encoder, "cuda"), config["text"],
                            batch=8, device="cuda", mask_codec=config["representation"]["mask_codec"])
record["selection"] = [{"letter": letters[e["patient"]], "patient": e["patient"], "kind": e["kind"],
                        "rank": ranked[e["kind"]].index(e["patient"]) + 1,
                        "slices": [{"frame_z": z, "category": c} for z, c in e["slices"]]} for e in selection]
record["skipped"] = [dict(s, rank=ranked[s["kind"]].index(s["patient"]) + 1) for s in skipped]
slice_table = [{"name": n, "label_px": {f"id{c}": int((t["label"] == c).sum()) for c in (1, 2, 3)},
                "region_px": int(t["region"].sum())} for n, t in zip(names, targets)]
record["slice_table"] = slice_table
print("skipped (rank, kind, reason):", [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]])
for row in slice_table:
    print(row)

## 5. Evaluation helper (fixed-noise loss, generations, scores, PNG grid)

In [ ]:
from rhuh_synthesis.overfit import (fixed_eval_loss, generate, save_overfit_grid, score_slices,
                                   shifted_conditions, summarize)

GENERATION = dict(num_inference_steps=config["sampling"]["num_inference_steps"],
                  guidance_scale=config["sampling"]["guidance_scale"], seed=S4["generation"]["seed"],
                  batch=S4["generation"]["batch_pairs"], mask_codec=config["representation"]["mask_codec"])


def scored(images, masks, step, suffix=""):
    rows = score_slices(images, masks, targets, pixel_mm=pixel_mm)
    panels = [{"name": n, "pre_image": t["pre_image"], "post_image": t["post_image"], "generated_image": i,
               "label": t["label"], "generated_mask": m} for n, t, i, m in zip(names, targets, images, masks)]
    save_overfit_grid(RUN / "grids" / f"step{step:06d}{suffix}.png", panels, title=f"Stage 4 {RUN_TAG} step {step}{suffix}")
    return {"slices": [dict(row, name=n) for row, n in zip(rows, names)], "summary": summarize(rows, categories)}


def evaluate(unet, step, shifted=False):
    start = time.perf_counter()
    result = {"step": step, "eval_loss": fixed_eval_loss(unet, cache, items, noise_scheduler,
                                                         timesteps=S4["eval_loss"]["timesteps"],
                                                         seed=S4["eval_loss"]["noise_seed"],
                                                         batch=S4["eval_loss"]["batch"])}
    result.update(scored(*generate(unet, vae, ddim, cache, items, **GENERATION), step))
    if shifted:
        conditions = shifted_conditions(items, patients)
        result["shifted"] = scored(*generate(unet, vae, ddim, cache, items, condition_items=conditions, **GENERATION),
                                   step, "_shifted")
    result["seconds"] = time.perf_counter() - start
    summary = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in result["summary"].items()}
    print(f"step {step}: loss image {result['eval_loss']['image']['mean']:.4f}, "
          f"mask {result['eval_loss']['mask']['mean']:.4f} | {summary} | {result['seconds']:.0f}s")
    if shifted:
        print("   shifted-condition control:", {k: (round(v, 4) if isinstance(v, float) else v)
                                                 for k, v in result["shifted"]["summary"].items()})
    return result

## 6. Step 0, then train 500 steps with evaluations at 250 and 500

In [ ]:
from rhuh_synthesis.model import expand_conv_in
from rhuh_synthesis.training import SliceSampler, slice_pools, train

RUN.mkdir(parents=True)
unet11 = expand_conv_in(copy.deepcopy(unet4), 11).to("cuda")
unet4.to("cpu")
torch.cuda.empty_cache()
optimizer = torch.optim.AdamW(unet11.parameters(), lr=config["training"]["learning_rate"],
                              weight_decay=config["training"]["weight_decay"])
sampler = SliceSampler({p: slice_pools(exports[p]["label"]) for p in patients}, seed=S4["train_seed"],
                       positive_pool_probability=config["training"]["positive_pool_probability"])
rng, generator = np.random.default_rng(S4["train_seed"]), torch.Generator().manual_seed(S4["train_seed"])
log_path = RUN / "train_log.jsonl"
torch.cuda.reset_peak_memory_stats()
evaluations = [evaluate(unet11, 0)]
step, train_seconds = 0, 0.0
for target in S4["evaluate_at"][1:]:
    start = time.perf_counter()
    step = train(unet11, cache, sampler, noise_scheduler, optimizer, steps=target - step,
                 accumulation=S4["accumulation"], microbatch=S4["microbatch_pairs"], rng=rng, generator=generator,
                 log_path=log_path, start_step=step, precision=S4["precision"], config_sha256=record["config_sha256"])
    train_seconds += time.perf_counter() - start
    print(f"trained to step {step} ({train_seconds:.0f}s of training so far)")
    evaluations.append(evaluate(unet11, step, shifted=target == S4["steps"]))
record["train_seconds"] = train_seconds
record["peak_gib"] = torch.cuda.max_memory_allocated() / 2**30
record["adapter_added_weight_abs_sum"] = float(unet11.conv_in.weight[:, 4:].detach().abs().sum())

## 7. Decision, write-once record and the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json
from rhuh_synthesis.overfit import stage4_decision

training_log = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
decision, checks = stage4_decision(training_log, S4["steps"], evaluations[0], evaluations[-1],
                                   evaluations[-1]["shifted"]["summary"], S4["criteria"])
record.update(evaluations=evaluations, training_log=training_log, sampler_counts=sampler.state_dict()["counts"],
              checks=checks, decision=decision, total_seconds=time.perf_counter() - STARTED)
path, digest = write_once_json(RUN / "record.json", record)
print("saved", path, digest[:12])
if SAVE_FINAL_CHECKPOINT:
    from rhuh_synthesis.training import save_checkpoint
    print(save_checkpoint(f"/content/stage4_{RUN_TAG}_step{step:06d}.pt", unet=unet11, optimizer=optimizer, step=step,
                          sampler=sampler, rng=rng, generator=generator, config_sha256=record["config_sha256"]))
losses = [line["loss"] for line in training_log]
shareable = {
    "commit": commit, "config_sha256": record["config_sha256"][:12], "gpu": record["gpu"], "torch": record["torch"],
    "checkpoint_sha256": record["checkpoint"]["sha256"][:12],
    "equivalence_tf32_off": record["equivalence_tf32_off"], "equivalence_tf32_on": record["equivalence_tf32_on"],
    "skipped": [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]],
    "slices": slice_table,
    "train_loss": {"first": losses[0], "last": losses[-1], "mean_first_50": float(np.mean(losses[:50])),
                   "mean_last_50": float(np.mean(losses[-50:])), "all_finite": all(line["finite"] for line in training_log)},
    "evaluations": [{"step": e["step"], "eval_loss": e["eval_loss"], "summary": e["summary"],
                     "slices": e["slices"], "seconds": round(e["seconds"])} for e in evaluations],
    "shifted": evaluations[-1]["shifted"],
    "adapter_added_weight_abs_sum": record["adapter_added_weight_abs_sum"],
    "train_seconds": round(record["train_seconds"]), "peak_gib": round(record["peak_gib"], 1),
    "checks": checks, "decision": decision,
}
print(json.dumps(shareable, indent=1))

## Next

Send Claude the printed JSON block. It has slice names (A to D) but no patient IDs. If you can, also download the PNG grids from `synthesis/EXP-2026-001/stage4/<RUN_TAG>/grids/` on Drive into the same folder on your laptop, so they can be reviewed locally. Stage 5 (the 1,000-step pilot) needs your explicit go-ahead after the review.